### Import environment

In [13]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["JAVA_HOME"] = os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = os.getenv("SPARK_HOME")

### Import necessary libraries

In [192]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### Create session Spark

In [16]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FootballES")
    .getOrCreate()
)

### Verify my session spark

In [17]:
print(spark)

### Analyze Data with Spark

In [217]:
df = spark.read.format("csv").options(
    header="true").load("transformed_data.csv")

In [218]:
df.show(5)

+-----+----------+------------+--------------------+---------+---------+
|Round|      Date|      Team 1|              Team 2|FT Team 1|FT Team 2|
+-----+----------+------------+--------------------+---------+---------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|        0|        0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|        2|        0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|        0|        2|
|    1|2020-09-13|FC Barcelona|            Elche CF|     NULL|     NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|     NULL|     NULL|
+-----+----------+------------+--------------------+---------+---------+
only showing top 5 rows


In [219]:
# Rename columns FT Team 1 and FT Team 2
df = df.selectExpr(
    "*",
    "`Team 1` as `HomeTeam`",
    "`Team 2` as `AwayTeam`",
    "`FT Team 1` as `HomeTeamGoals`",
    "`FT Team 2` as `AwayTeamGoals`"
)

In [220]:
df.show(5)

+-----+----------+------------+--------------------+---------+---------+------------+--------------------+-------------+-------------+
|Round|      Date|      Team 1|              Team 2|FT Team 1|FT Team 2|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|
+-----+----------+------------+--------------------+---------+---------+------------+--------------------+-------------+-------------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|        0|        0|    SD Eibar|       RC Celta Vigo|            0|            0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|        2|        0|  Granada CF|Athletic Club Bilbao|            2|            0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|        0|        2|    Cádiz CF|          CA Osasuna|            0|            2|
|    1|2020-09-13|FC Barcelona|            Elche CF|     NULL|     NULL|FC Barcelona|            Elche CF|         NULL|         NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|   

In [221]:
# drop original columns
df = df.drop("FT Team 1", "FT Team 2", "Team 1", "Team 2")

In [222]:
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|
+-----+----------+------------+--------------------+-------------+-------------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|
+-----+----------+------------+--------------------+-------------+-------------+
only showing top 5 rows


In [223]:
df = df.withColumn("Results",
                   F.when(F.col("HomeTeamGoals") > F.col(
                       "AwayTeamGoals"), "HomeTeamWin")
                   .when(F.col("HomeTeamGoals") < F.col("AwayTeamGoals"), "AwayTeamWin")
                   .otherwise("Draw"))

In [200]:
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+-----------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|       Draw|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
only showing top 5 rows


In [224]:
df = (df
      .withColumn("Season", F.substring(F.col("date"), 1, 4))
      )

In [225]:
df.limit(10).show()

+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|
|    1|2020-09-13|      FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|  2020|
|    1|2020-09-13|       Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|  2020|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|
|    1|2020-09-13|Real Valladolid CF|       Re

In [226]:
# Delete rows where HomeTeamGoals or AwayTeamGoals is null
df = df.dropna(subset=["HomeTeamGoals", "AwayTeamGoals"])

In [227]:
df.show(5)

+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|
|    1|2020-09-13|Real Valladolid CF|       Real Sociedad|            1|            1|       Draw|  2020|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+
only showing top 5 rows


In [228]:
df = df.withColumn("HomeTeamWin", F.when(F.col("Results") == "HomeTeamWin", 1).otherwise(0)) \
       .withColumn("AwayTeamWin", F.when(F.col("Results") == "AwayTeamWin", 1).otherwise(0)) \
       .withColumn("GameTie", F.when(F.col("Results") == "Draw", 1).otherwise(0))

In [229]:
df.limit(10).show()

+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|HomeTeamWin|AwayTeamWin|GameTie|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|          0|          0|      1|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|          1|          0|      0|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|          0|          1|      0|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|          0|          1|      0|
|    1|2020-09-13|Real Valladolid 

In [230]:
df.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: string (nullable = true)
 |-- AwayTeamGoals: string (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: string (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [231]:
# Types: caster les colonnes en place sans créer de nouvelles colonnes
df = df.withColumn("Date", F.col("Date").cast("date")) \
    .withColumn("HomeTeamGoals", F.col("HomeTeamGoals").cast("int")) \
    .withColumn("AwayTeamGoals", F.col("AwayTeamGoals").cast("int")) \
    .withColumn("Season", F.col("Season").cast("int"))

In [232]:
df.limit(10).show()

+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|HomeTeamWin|AwayTeamWin|GameTie|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|          0|          0|      1|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|          1|          0|      0|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|          0|          1|      0|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|          0|          1|      0|
|    1|2020-09-13|Real Valladolid 

In [233]:
df.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: integer (nullable = true)
 |-- AwayTeamGoals: integer (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: integer (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [234]:
# Delete unecessay columns
df = df.drop("Date", "Round")

In [235]:
df.show(5)

+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|HomeTeamWin|AwayTeamWin|GameTie|
+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|          0|          0|      1|
|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|          1|          0|      0|
|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|          0|          1|      0|
|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|          0|          1|      0|
|Real Valladolid CF|       Real Sociedad|            1|            1|       Draw|  2020|          0|          0|      1|
+------------------+------------

In [ ]:
# Create dataframe containing statistics on home matches
df_home_matches = df.groupBy('Season', 'HomeTeam') \
                    .agg(F.sum('HomeTeamWin').alias('TotalHomeWin'),
                         F.sum('AwayTeamWin').alias('TotalHomeLoss'),
                         F.sum('GameTie').alias('TotalHomeTie'),
                         F.sum('HomeTeamGoals').alias('HomeScoredGoals'),
                         F.sum('AwayTeamGoals').alias('HomeAgainstGoals')
                         ).withColumnRenamed('HomeTeam', 'Team')

In [239]:
df_home_matches.show()

+------+--------------------+------------+-------------+------------+---------------+----------------+
|Season|                Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+
|  2016|Athletic Club Bilbao|          12|            3|           4|             36|              19|
|  2017|       Villarreal CF|           9|            5|           4|             32|              19|
|  2015|          Levante UD|           5|            8|           7|             21|              26|
|  2014|          Granada CF|           6|            7|           5|             14|              18|
|  2017|       RC Celta Vigo|           7|            8|           4|             26|              27|
|  2018|        FC Barcelona|          15|            1|           4|             57|              19|
|  2014|          Levante UD|           7|            7|           5|    

In [ ]:
# Create dataframe containing statistics on away matches
df_away_matches = df.groupBy('Season', 'AwayTeam') \
                    .agg(F.sum('AwayTeamWin').alias('TotalAwayWin'),
                         F.sum('HomeTeamWin').alias('TotalAwayLoss'),
                         F.sum('GameTie').alias('TotalAwayTie'),
                         F.sum('AwayTeamGoals').alias('AwayScoredGoals'),
                         F.sum('HomeTeamGoals').alias('AwayAgainstGoals')
                         ).withColumnRenamed('AwayTeam', 'Team')

In [242]:
df_away_matches.limit(10).show()

+------+--------------------+------------+-------------+------------+---------------+----------------+
|Season|                Team|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+
|  2016|Athletic Club Bilbao|           6|            9|           3|             19|              27|
|  2017|       Villarreal CF|          10|            7|           4|             23|              23|
|  2015|          Levante UD|           3|           14|           2|             13|              41|
|  2014|          Granada CF|           2|           12|           5|             13|              42|
|  2017|       RC Celta Vigo|           6|           12|           2|             32|              36|
|  2018|        FC Barcelona|          10|            2|           6|             45|              22|
|  2014|          Levante UD|           3|            7|           8|    

In [ ]:
# join home and away dataframes
df_team_stats = df_home_matches.join(
    df_away_matches, ['Season', 'Team'], 'inner')